In [ ]:
# =====================================================
# HOSPITAL MANAGEMENT SYSTEM – COMPLETE WORKING VERSION
# =====================================================

import sys, subprocess, importlib, threading, os, datetime as dt
from datetime import datetime, timedelta

# Auto-install missing packages
required_pkgs = ['pyodbc', 'customtkinter', 'pillow', 'matplotlib', 'pandas', 'openpyxl', 'fpdf']
for pkg in required_pkgs:
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

import tkinter as tk
from tkinter import ttk, messagebox
import pyodbc
import customtkinter as ctk
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
import pandas as pd
from fpdf import FPDF
from PIL import Image

# ========== DATABASE CONNECTION ==========
# Change this if your SQL Server instance is different (e.g., "localhost" for default instance)
SERVER = r"localhost\SQLEXPRESS"   # For SQL Server Express
DATABASE = "Hospital_Final_Project"

def get_connection_string():
    for driver in ["{ODBC Driver 18 for SQL Server}", "{ODBC Driver 17 for SQL Server}", 
                   "{ODBC Driver 13 for SQL Server}", "{SQL Server}"]:
        conn_str = f"DRIVER={driver};SERVER={SERVER};DATABASE={DATABASE};Trusted_Connection=yes;TrustServerCertificate=yes;"
        try:
            conn = pyodbc.connect(conn_str, timeout=5)
            conn.close()
            return conn_str
        except:
            continue
    raise Exception("No suitable ODBC driver found. Check SQL Server and driver installation.")

CONN_STR = get_connection_string()
print("Connected to database")

def query_db(query, params=(), fetch_all=True):
    conn = pyodbc.connect(CONN_STR)
    cursor = conn.cursor()
    cursor.execute(query, params)
    is_select = query.strip().upper().startswith(('SELECT', 'WITH'))
    if is_select:
        result = cursor.fetchall() if fetch_all else cursor.fetchone()
    else:
        result = None
    conn.commit()
    conn.close()
    return result

def sanitize_row(row):
    return [str(v).replace('(', '').replace(')', '').replace(',', ' ') if v is not None else "" for v in row]

def export_to_excel(treeview, filename):
    data = []
    columns = [treeview.heading(c)['text'] for c in treeview['columns']]
    for child in treeview.get_children():
        row = treeview.item(child)['values']
        data.append(row)
    df = pd.DataFrame(data, columns=columns)
    df.to_excel(filename, index=False)
    messagebox.showinfo("Export", f"Exported to {filename}")

def generate_prescription_pdf(patient_name, doctor_name, diagnosis, medicines, advice, followup):
    pdf = FPDF()
    pdf.add_page()
    pdf.set_font("Arial", 'B', 16)
    pdf.cell(200, 10, "Prescription", ln=1, align='C')
    pdf.ln(10)
    pdf.set_font("Arial", size=12)
    pdf.cell(200, 10, f"Patient: {patient_name}", ln=1)
    pdf.cell(200, 10, f"Doctor: {doctor_name}", ln=1)
    pdf.cell(200, 10, f"Diagnosis: {diagnosis}", ln=1)
    pdf.multi_cell(0, 10, f"Medicines:\n{medicines}")
    pdf.multi_cell(0, 10, f"Advice:\n{advice}")
    if followup:
        pdf.cell(200, 10, f"Follow-up: {followup}", ln=1)
    filename = f"prescription_{patient_name.replace(' ', '_')}_{datetime.now().strftime('%Y%m%d%H%M%S')}.pdf"
    pdf.output(filename)
    return filename

# ========== LOGIN WINDOW ==========
class LoginWindow:
    def __init__(self):
        self.root = ctk.CTk()
        self.root.title("Hospital Management System - Login")
        self.root.geometry("450x450")
        ctk.set_appearance_mode("dark")
        ctk.set_default_color_theme("blue")
        
        try:
            logo_image = ctk.CTkImage(light_image=Image.open("hospital_logo.png"), size=(100, 100))
            ctk.CTkLabel(self.root, image=logo_image, text="").pack(pady=20)
        except:
            ctk.CTkLabel(self.root, text="🏥", font=("Arial", 60)).pack(pady=20)
        
        frame = ctk.CTkFrame(self.root)
        frame.pack(pady=20, padx=50, fill="both", expand=True)
        
        ctk.CTkLabel(frame, text="Hospital Login", font=("Arial", 24, "bold")).pack(pady=20)
        ctk.CTkLabel(frame, text="Username").pack(anchor="w")
        self.username = ctk.CTkEntry(frame, width=250)
        self.username.pack(pady=5)
        ctk.CTkLabel(frame, text="Password").pack(anchor="w")
        self.password = ctk.CTkEntry(frame, width=250, show="*")
        self.password.pack(pady=5)
        ctk.CTkButton(frame, text="Login", command=self.do_login).pack(pady=20)
        self.password.bind("<Return>", lambda e: self.do_login())
        
        self.root.mainloop()
    
    def do_login(self):
        uname = self.username.get().strip()
        pwd = self.password.get().strip()
        user = query_db("SELECT u.UserID, u.RoleID, r.RoleName, u.RelatedID FROM Users u JOIN Roles r ON u.RoleID=r.RoleID WHERE u.Username=? AND u.PasswordHash=?", (uname, pwd), fetch_all=False)
        if user:
            self.root.destroy()
            role = user[2]
            related_id = user[3]
            app = HospitalApp(uname, role, related_id)
        else:
            messagebox.showerror("Error", "Invalid username or password")

# ========== MAIN APPLICATION ==========
class HospitalApp:
    def __init__(self, username, role, related_id):
        self.username = username
        self.role = role
        self.related_id = related_id
        self.root = ctk.CTk()
        self.root.title(f"Hospital Management System - {role}")
        self.root.geometry("1400x800")
        
        self.status_bar = ctk.CTkLabel(self.root, text=f"Logged in as {username} ({role})", height=30, fg_color="#2b2b2b", anchor="w")
        self.status_bar.pack(side="bottom", fill="x")
        
        header = ctk.CTkFrame(self.root, height=80, corner_radius=0, fg_color="#1f538d")
        header.pack(fill="x")
        try:
            logo_img = ctk.CTkImage(light_image=Image.open("hospital_logo.png"), size=(50, 50))
            ctk.CTkLabel(header, image=logo_img, text="").pack(side="left", padx=10)
        except:
            ctk.CTkLabel(header, text="🏥", font=("Arial", 40)).pack(side="left", padx=10)
        ctk.CTkLabel(header, text="Hospital Management System", font=("Arial", 24, "bold"), text_color="white").pack(side="left", padx=20, pady=20)
        
        main_container = ctk.CTkFrame(self.root)
        main_container.pack(fill="both", expand=True, padx=10, pady=10)
        
        self.tabview = ctk.CTkTabview(main_container)
        self.tabview.pack(fill="both", expand=True)
        
        if role == "Admin":
            self.add_admin_tabs()
        elif role == "Doctor":
            self.add_doctor_tabs()
        elif role == "Receptionist":
            self.add_receptionist_tabs()
        else:
            self.add_patient_tabs()
        
        self.refresh_all()
        self.root.mainloop()
    
    def add_admin_tabs(self):
        self.tabview.add("Dashboard")
        self.tabview.add("Patients")
        self.tabview.add("Doctors")
        self.tabview.add("Nurses")
        self.tabview.add("Appointments")
        self.tabview.add("Prescriptions")
        self.tabview.add("Billing")
        self.tabview.add("Bed/Ward")
        self.tabview.add("Laboratory")
        self.tabview.add("Pharmacy")
        self.tabview.add("Audit Logs")
        self.tabview.add("Users")
        self.setup_dashboard()
        self.setup_patients()
        self.setup_doctors()
        self.setup_nurses()
        self.setup_appointments()
        self.setup_prescriptions()
        self.setup_billing()
        self.setup_bed_ward()
        self.setup_laboratory()
        self.setup_pharmacy()
        self.setup_audit_logs()
        self.setup_users()
    
    def add_doctor_tabs(self):
        self.tabview.add("My Schedule")
        self.tabview.add("Appointments")
        self.tabview.add("Prescriptions")
        self.tabview.add("Patient History")
        self.setup_doctor_schedule()
        self.setup_appointments()
        self.setup_prescriptions()
        self.setup_patient_history()
    
    def add_receptionist_tabs(self):
        self.tabview.add("Dashboard")
        self.tabview.add("Patients")
        self.tabview.add("Appointments")
        self.tabview.add("Billing")
        self.tabview.add("Admissions")
        self.setup_dashboard()
        self.setup_patients()
        self.setup_appointments()
        self.setup_billing()
        self.setup_admissions()
    
    def add_patient_tabs(self):
        self.tabview.add("My Appointments")
        self.tabview.add("My Prescriptions")
        self.tabview.add("My Bills")
        self.setup_patient_appointments()
        self.setup_patient_prescriptions()
        self.setup_patient_bills()
    
    def refresh_all(self):
        if self.role == "Admin":
            self.refresh_patients()
            self.refresh_doctors()
            self.refresh_nurses()
            self.refresh_appointments()
            self.refresh_patient_doctor_combos()
            self.refresh_completed_appointments()
            self.update_dashboard()
            self.refresh_beds()
            self.refresh_lab_orders()
            self.refresh_medicines()
            self.refresh_audit()
            self.refresh_users()
        elif self.role == "Doctor":
            self.refresh_doctor_schedule()
            self.refresh_appointments()
            self.refresh_patient_history()
        elif self.role == "Receptionist":
            self.refresh_patients()
            self.refresh_appointments()
            self.update_dashboard()
            self.refresh_admissions()
        else:
            self.refresh_patient_appointments()
            self.refresh_patient_prescriptions()
            self.refresh_patient_bills()
        self.status_bar.configure(text="Data refreshed")
    
    # ---------- DASHBOARD ----------
    def setup_dashboard(self):
        tab = self.tabview.tab("Dashboard")
        stats_frame = ctk.CTkFrame(tab)
        stats_frame.pack(fill="x", padx=20, pady=20)
        
        self.patient_count_label = ctk.CTkLabel(stats_frame, text="Patients: 0", font=("Arial", 18, "bold"), fg_color="#2b5a8c", corner_radius=10, width=200, height=80)
        self.patient_count_label.pack(side="left", padx=20, pady=10)
        self.doctor_count_label = ctk.CTkLabel(stats_frame, text="Doctors: 0", font=("Arial", 18, "bold"), fg_color="#2b5a8c", corner_radius=10, width=200, height=80)
        self.doctor_count_label.pack(side="left", padx=20, pady=10)
        self.appointment_count_label = ctk.CTkLabel(stats_frame, text="Today's Appointments: 0", font=("Arial", 18, "bold"), fg_color="#2b5a8c", corner_radius=10, width=280, height=80)
        self.appointment_count_label.pack(side="left", padx=20, pady=10)
        self.revenue_label = ctk.CTkLabel(stats_frame, text="Today's Revenue: $0", font=("Arial", 18, "bold"), fg_color="#2b5a8c", corner_radius=10, width=250, height=80)
        self.revenue_label.pack(side="left", padx=20, pady=10)
        
        chart_frame = ctk.CTkFrame(tab)
        chart_frame.pack(fill="both", expand=True, padx=20, pady=20)
        self.fig, self.ax = plt.subplots(figsize=(6, 4), facecolor="#2b2b2b")
        self.ax.set_facecolor("#2b2b2b")
        self.ax.tick_params(colors="white")
        self.ax.set_title("Appointments per Day (Last 7 Days)", color="white")
        self.canvas = FigureCanvasTkAgg(self.fig, master=chart_frame)
        self.canvas.get_tk_widget().pack(fill="both", expand=True)
        
        upcoming_frame = ctk.CTkFrame(tab)
        upcoming_frame.pack(fill="both", expand=True, padx=20, pady=10)
        ctk.CTkLabel(upcoming_frame, text="Upcoming Appointments", font=("Arial", 16, "bold")).pack(anchor="w", padx=10)
        self.upcoming_tree = ttk.Treeview(upcoming_frame, columns=("Date","Patient","Doctor","Time"), show="headings", height=6)
        for c in ("Date","Patient","Doctor","Time"):
            self.upcoming_tree.heading(c, text=c)
        self.upcoming_tree.pack(fill="both", expand=True, padx=10, pady=5)
        self.update_dashboard()
    
    def update_dashboard(self):
        patients = query_db("SELECT COUNT(*) FROM Patients")
        if patients: self.patient_count_label.configure(text=f"Patients: {patients[0][0]}")
        doctors = query_db("SELECT COUNT(*) FROM Doctors")
        if doctors: self.doctor_count_label.configure(text=f"Doctors: {doctors[0][0]}")
        today = datetime.today().strftime("%Y-%m-%d")
        today_apps = query_db("SELECT COUNT(*) FROM Appointments WHERE AppointmentDate=?", (today,))
        if today_apps: self.appointment_count_label.configure(text=f"Today's Appointments: {today_apps[0][0]}")
        revenue = query_db("SELECT SUM(PaidAmount) FROM Bills WHERE PaymentDate=?", (today,))
        if revenue and revenue[0][0]:
            self.revenue_label.configure(text=f"Today's Revenue: ${revenue[0][0]:.2f}")
        else: self.revenue_label.configure(text="Today's Revenue: $0")
        
        dates, counts = [], []
        for i in range(6, -1, -1):
            d = datetime.today() - timedelta(days=i)
            cnt = query_db("SELECT COUNT(*) FROM Appointments WHERE AppointmentDate=?", (d.strftime("%Y-%m-%d"),))
            dates.append(d.strftime("%m/%d"))
            counts.append(cnt[0][0] if cnt else 0)
        self.ax.clear()
        self.ax.bar(dates, counts, color="#1f538d")
        self.ax.set_title("Appointments per Day (Last 7 Days)", color="white")
        self.ax.set_xlabel("Date", color="white")
        self.ax.set_ylabel("Count", color="white")
        self.ax.tick_params(colors="white")
        self.fig.tight_layout()
        self.canvas.draw()
        
        for row in self.upcoming_tree.get_children():
            self.upcoming_tree.delete(row)
        upcoming = query_db("""SELECT TOP 10 AppointmentDate, p.FullName, d.FullName, StartTime
                               FROM Appointments a JOIN Patients p ON a.PatientID=p.PatientID JOIN Doctors d ON a.DoctorID=d.DoctorID
                               WHERE AppointmentDate >= ? AND Status='Scheduled' ORDER BY AppointmentDate""", (today,))
        if upcoming:
            for u in upcoming: self.upcoming_tree.insert("", "end", values=sanitize_row(u))
    
    # ---------- PATIENTS ----------
    def setup_patients(self):
        tab = self.tabview.tab("Patients")
        top = ctk.CTkFrame(tab)
        top.pack(fill="x", padx=10, pady=10)
        ctk.CTkLabel(top, text="Search:").pack(side="left", padx=5)
        self.patient_search = ctk.CTkEntry(top, width=200)
        self.patient_search.pack(side="left", padx=5)
        self.patient_search.bind("<KeyRelease>", lambda e: self.refresh_patients())
        ctk.CTkButton(top, text="➕ Add Patient", command=self.add_patient).pack(side="right", padx=5)
        
        self.patient_tree = ttk.Treeview(tab, columns=("ID","Name","Phone","Email","BloodGroup"), show="headings")
        for c in ("ID","Name","Phone","Email","BloodGroup"):
            self.patient_tree.heading(c, text=c)
        self.patient_tree.pack(fill="both", expand=True, padx=10, pady=5)
        
        btn_frame = ctk.CTkFrame(tab)
        btn_frame.pack(fill="x", padx=10, pady=10)
        ctk.CTkButton(btn_frame, text="✏️ Edit", command=self.edit_patient).pack(side="left", padx=5)
        ctk.CTkButton(btn_frame, text="🗑️ Delete", command=self.delete_patient).pack(side="left", padx=5)
        ctk.CTkButton(btn_frame, text="📊 Export to Excel", command=lambda: export_to_excel(self.patient_tree, "patients.xlsx")).pack(side="left", padx=5)
        self.refresh_patients()
    
    def refresh_patients(self):
        for row in self.patient_tree.get_children():
            self.patient_tree.delete(row)
        search = self.patient_search.get().strip()
        q = "SELECT PatientID, FullName, Phone, Email, BloodGroup FROM Patients"
        if search:
            q += f" WHERE FullName LIKE '%{search}%' OR Phone LIKE '%{search}%'"
        rows = query_db(q)
        for r in rows:
            self.patient_tree.insert("", "end", values=sanitize_row(r))
    
    def add_patient(self):
        self.patient_form()
    
    def edit_patient(self):
        sel = self.patient_tree.selection()
        if not sel:
            messagebox.showwarning("Warning", "Select a patient")
            return
        pid = self.patient_tree.item(sel[0])['values'][0]
        self.patient_form(pid)
    
    def patient_form(self, pid=None):
        win = ctk.CTkToplevel(self.root)
        win.title("Patient Form")
        win.geometry("500x600")
        fields = {}
        labels = ["Full Name","Date of Birth (YYYY-MM-DD)","Gender","Phone","Email","Address","Blood Group"]
        for i, lbl in enumerate(labels):
            ctk.CTkLabel(win, text=lbl).grid(row=i, column=0, padx=10, pady=5, sticky="w")
            e = ctk.CTkEntry(win, width=300)
            e.grid(row=i, column=1, padx=10, pady=5)
            fields[lbl] = e
        if pid:
            data = query_db("SELECT FullName,DateOfBirth,Gender,Phone,Email,Address,BloodGroup FROM Patients WHERE PatientID=?", (pid,), fetch_all=False)
            if data:
                fields["Full Name"].insert(0, data[0])
                fields["Date of Birth (YYYY-MM-DD)"].insert(0, str(data[1]))
                fields["Gender"].insert(0, data[2])
                fields["Phone"].insert(0, data[3])
                fields["Email"].insert(0, data[4])
                fields["Address"].insert(0, data[5])
                fields["Blood Group"].insert(0, data[6])
        def save():
            vals = [fields[l].get().strip() for l in labels]
            if not vals[0] or not vals[3]:
                messagebox.showerror("Error", "Name and Phone required")
                return
            if pid:
                query_db("UPDATE Patients SET FullName=?,DateOfBirth=?,Gender=?,Phone=?,Email=?,Address=?,BloodGroup=? WHERE PatientID=?", (*vals, pid), fetch_all=False)
            else:
                query_db("INSERT INTO Patients (FullName,DateOfBirth,Gender,Phone,Email,Address,BloodGroup) VALUES (?,?,?,?,?,?,?)", vals, fetch_all=False)
            self.refresh_patients()
            win.destroy()
            messagebox.showinfo("Success", "Patient saved")
        ctk.CTkButton(win, text="Save", command=save).grid(row=len(labels), column=0, columnspan=2, pady=20)
    
    def delete_patient(self):
        sel = self.patient_tree.selection()
        if not sel: return
        if messagebox.askyesno("Confirm", "Delete patient?"):
            pid = self.patient_tree.item(sel[0])['values'][0]
            query_db("DELETE FROM Patients WHERE PatientID=?", (pid,), fetch_all=False)
            self.refresh_patients()
    
    # ---------- DOCTORS ----------
    def setup_doctors(self):
        tab = self.tabview.tab("Doctors")
        top = ctk.CTkFrame(tab)
        top.pack(fill="x", padx=10, pady=10)
        ctk.CTkButton(top, text="➕ Add Doctor", command=self.add_doctor).pack(side="right", padx=5)
        
        self.doctor_tree = ttk.Treeview(tab, columns=("ID","Name","Specialization","Phone","Fee"), show="headings")
        for c in ("ID","Name","Specialization","Phone","Fee"):
            self.doctor_tree.heading(c, text=c)
        self.doctor_tree.pack(fill="both", expand=True, padx=10, pady=5)
        
        btn_frame = ctk.CTkFrame(tab)
        btn_frame.pack(fill="x", padx=10, pady=10)
        ctk.CTkButton(btn_frame, text="✏️ Edit", command=self.edit_doctor).pack(side="left", padx=5)
        ctk.CTkButton(btn_frame, text="🗑️ Delete", command=self.delete_doctor).pack(side="left", padx=5)
        ctk.CTkButton(btn_frame, text="📅 Set Availability", command=self.open_availability).pack(side="left", padx=5)
        self.refresh_doctors()
    
    def refresh_doctors(self):
        for row in self.doctor_tree.get_children():
            self.doctor_tree.delete(row)
        rows = query_db("SELECT d.DoctorID, d.FullName, s.SpecializationName, d.Phone, d.ConsultationFee FROM Doctors d JOIN Specializations s ON d.SpecializationID=s.SpecializationID")
        for r in rows:
            self.doctor_tree.insert("", "end", values=sanitize_row(r))
    
    def add_doctor(self):
        self.doctor_form()
    
    def edit_doctor(self):
        sel = self.doctor_tree.selection()
        if not sel:
            messagebox.showwarning("Warning", "Select a doctor")
            return
        did = self.doctor_tree.item(sel[0])['values'][0]
        self.doctor_form(did)
    
    def doctor_form(self, did=None):
        win = ctk.CTkToplevel(self.root)
        win.title("Doctor Form")
        win.geometry("500x600")
        fields = {}
        labels = ["Full Name","Phone","Email","Qualification","Consultation Fee"]
        for i, lbl in enumerate(labels):
            ctk.CTkLabel(win, text=lbl).grid(row=i, column=0, padx=10, pady=5, sticky="w")
            e = ctk.CTkEntry(win, width=300)
            e.grid(row=i, column=1, padx=10, pady=5)
            fields[lbl] = e
        ctk.CTkLabel(win, text="Specialization").grid(row=len(labels), column=0, padx=10, pady=5, sticky="w")
        specs = ["Cardiologist","Dermatologist","Neurologist","Pediatrician","Orthopedic","General Physician"]
        spec_var = tk.StringVar(value=specs[0])
        spec_combo = ttk.Combobox(win, textvariable=spec_var, values=specs, state="readonly", width=30)
        spec_combo.grid(row=len(labels), column=1, padx=10, pady=5)
        if did:
            data = query_db("SELECT d.FullName, d.Phone, d.Email, d.Qualification, d.ConsultationFee, s.SpecializationName FROM Doctors d JOIN Specializations s ON d.SpecializationID=s.SpecializationID WHERE d.DoctorID=?", (did,), fetch_all=False)
            if data:
                fields["Full Name"].insert(0, data[0])
                fields["Phone"].insert(0, data[1])
                fields["Email"].insert(0, data[2])
                fields["Qualification"].insert(0, data[3])
                fields["Consultation Fee"].insert(0, str(data[4]))
                spec_var.set(data[5])
        def save():
            vals = [fields[l].get().strip() for l in labels]
            spec = spec_var.get()
            if not vals[0] or not spec:
                messagebox.showerror("Error", "Name and specialization required")
                return
            spec_row = query_db("SELECT SpecializationID FROM Specializations WHERE SpecializationName=?", (spec,))
            if spec_row:
                spec_id = spec_row[0][0]
            else:
                query_db("INSERT INTO Specializations (SpecializationName) VALUES (?)", (spec,), fetch_all=False)
                spec_id = query_db("SELECT SpecializationID FROM Specializations WHERE SpecializationName=?", (spec,), fetch_all=False)[0][0]
            fee = float(vals[4]) if vals[4] else 0
            if did:
                query_db("UPDATE Doctors SET FullName=?, SpecializationID=?, Phone=?, Email=?, Qualification=?, ConsultationFee=? WHERE DoctorID=?", (vals[0], spec_id, vals[1], vals[2], vals[3], fee, did), fetch_all=False)
            else:
                query_db("INSERT INTO Doctors (FullName,SpecializationID,Phone,Email,Qualification,ConsultationFee) VALUES (?,?,?,?,?,?)", (vals[0], spec_id, vals[1], vals[2], vals[3], fee), fetch_all=False)
            self.refresh_doctors()
            win.destroy()
            messagebox.showinfo("Success", "Doctor saved")
        ctk.CTkButton(win, text="Save", command=save).grid(row=len(labels)+1, column=0, columnspan=2, pady=20)
    
    def delete_doctor(self):
        sel = self.doctor_tree.selection()
        if not sel: return
        if messagebox.askyesno("Confirm", "Delete doctor?"):
            did = self.doctor_tree.item(sel[0])['values'][0]
            query_db("DELETE FROM Doctors WHERE DoctorID=?", (did,), fetch_all=False)
            self.refresh_doctors()
    
    def open_availability(self):
        sel = self.doctor_tree.selection()
        if not sel:
            messagebox.showwarning("Warning", "Select a doctor")
            return
        doc_id = self.doctor_tree.item(sel[0])['values'][0]
        doc_name = self.doctor_tree.item(sel[0])['values'][1]
        win = ctk.CTkToplevel(self.root)
        win.title(f"Availability for {doc_name}")
        win.geometry("400x450")
        
        ctk.CTkLabel(win, text="Day of Week:").pack(pady=5)
        days = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
        day_var = tk.StringVar(value=days[0])
        day_menu = ctk.CTkOptionMenu(win, variable=day_var, values=days)
        day_menu.pack(pady=5)
        
        ctk.CTkLabel(win, text="Start Time (HH:MM):").pack(pady=5)
        start_entry = ctk.CTkEntry(win, width=150)
        start_entry.pack()
        start_entry.insert(0, "09:00")
        
        ctk.CTkLabel(win, text="End Time (HH:MM):").pack(pady=5)
        end_entry = ctk.CTkEntry(win, width=150)
        end_entry.pack()
        end_entry.insert(0, "17:00")
        
        ctk.CTkLabel(win, text="Slot Duration (minutes):").pack(pady=5)
        dur_entry = ctk.CTkEntry(win, width=100)
        dur_entry.pack()
        dur_entry.insert(0, "30")
        
        def save():
            day_name = day_var.get()
            day_map = {"Monday":0,"Tuesday":1,"Wednesday":2,"Thursday":3,"Friday":4,"Saturday":5,"Sunday":6}
            day_int = day_map[day_name]
            start = start_entry.get().strip()
            end = end_entry.get().strip()
            dur = int(dur_entry.get().strip()) if dur_entry.get().strip() else 30
            if not start or not end:
                messagebox.showerror("Error", "Enter times")
                return
            query_db("INSERT INTO DoctorAvailability (DoctorID, DayOfWeek, StartTime, EndTime, SlotDuration) VALUES (?,?,?,?,?)", (doc_id, day_int, start, end, dur), fetch_all=False)
            messagebox.showinfo("Success", f"Availability added for {day_name}")
            win.destroy()
        
        ctk.CTkButton(win, text="Save", command=save).pack(pady=20)
    
    # ---------- NURSES ----------
    def setup_nurses(self):
        tab = self.tabview.tab("Nurses")
        top = ctk.CTkFrame(tab)
        top.pack(fill="x", padx=10, pady=10)
        ctk.CTkButton(top, text="➕ Add Nurse", command=self.add_nurse).pack(side="right", padx=5)
        
        self.nurse_tree = ttk.Treeview(tab, columns=("ID","Name","Phone","Email","Doctor"), show="headings")
        for c in ("ID","Name","Phone","Email","Doctor"):
            self.nurse_tree.heading(c, text=c)
        self.nurse_tree.pack(fill="both", expand=True, padx=10, pady=5)
        
        btn_frame = ctk.CTkFrame(tab)
        btn_frame.pack(fill="x", padx=10, pady=10)
        ctk.CTkButton(btn_frame, text="✏️ Edit", command=self.edit_nurse).pack(side="left", padx=5)
        ctk.CTkButton(btn_frame, text="🗑️ Delete", command=self.delete_nurse).pack(side="left", padx=5)
        self.refresh_nurses()
    
    def refresh_nurses(self):
        for row in self.nurse_tree.get_children():
            self.nurse_tree.delete(row)
        rows = query_db("SELECT n.NurseID, n.FullName, n.Phone, n.Email, d.FullName FROM Nurses n LEFT JOIN Doctors d ON n.AssignedDoctorID = d.DoctorID")
        for r in rows:
            self.nurse_tree.insert("", "end", values=sanitize_row(r))
    
    def add_nurse(self):
        self.nurse_form()
    
    def edit_nurse(self):
        sel = self.nurse_tree.selection()
        if not sel:
            messagebox.showwarning("Warning", "Select a nurse")
            return
        nid = self.nurse_tree.item(sel[0])['values'][0]
        self.nurse_form(nid)
    
    def nurse_form(self, nid=None):
        win = ctk.CTkToplevel(self.root)
        win.title("Nurse Form")
        win.geometry("500x500")
        fields = {}
        labels = ["Full Name","Phone","Email"]
        for i, lbl in enumerate(labels):
            ctk.CTkLabel(win, text=lbl).grid(row=i, column=0, padx=10, pady=5, sticky="w")
            e = ctk.CTkEntry(win, width=300)
            e.grid(row=i, column=1, padx=10, pady=5)
            fields[lbl] = e
        ctk.CTkLabel(win, text="Assign to Doctor (optional)").grid(row=len(labels), column=0, padx=10, pady=5, sticky="w")
        docs = query_db("SELECT DoctorID, FullName FROM Doctors")
        doc_list = [""] + [f"{d[0]} - {d[1]}" for d in docs]
        doc_var = tk.StringVar()
        doc_combo = ttk.Combobox(win, textvariable=doc_var, values=doc_list, state="readonly", width=30)
        doc_combo.grid(row=len(labels), column=1, padx=10, pady=5)
        if nid:
            data = query_db("SELECT FullName, Phone, Email, AssignedDoctorID FROM Nurses WHERE NurseID=?", (nid,), fetch_all=False)
            if data:
                fields["Full Name"].insert(0, data[0])
                fields["Phone"].insert(0, data[1])
                fields["Email"].insert(0, data[2])
                if data[3]:
                    doc_name = query_db("SELECT FullName FROM Doctors WHERE DoctorID=?", (data[3],), fetch_all=False)[0][0]
                    doc_var.set(f"{data[3]} - {doc_name}")
        def save():
            name = fields["Full Name"].get().strip()
            if not name:
                messagebox.showerror("Error", "Name required")
                return
            phone = fields["Phone"].get().strip()
            email = fields["Email"].get().strip()
            doc_val = doc_var.get()
            assigned = None
            if doc_val and "-" in doc_val:
                assigned = doc_val.split(" - ")[0]
            if nid:
                query_db("UPDATE Nurses SET FullName=?, Phone=?, Email=?, AssignedDoctorID=? WHERE NurseID=?", (name, phone, email, assigned, nid), fetch_all=False)
            else:
                query_db("INSERT INTO Nurses (FullName, Phone, Email, AssignedDoctorID) VALUES (?,?,?,?)", (name, phone, email, assigned), fetch_all=False)
            self.refresh_nurses()
            win.destroy()
            messagebox.showinfo("Success", "Nurse saved")
        ctk.CTkButton(win, text="Save", command=save).grid(row=len(labels)+1, column=0, columnspan=2, pady=20)
    
    def delete_nurse(self):
        sel = self.nurse_tree.selection()
        if not sel: return
        if messagebox.askyesno("Confirm", "Delete nurse?"):
            nid = self.nurse_tree.item(sel[0])['values'][0]
            query_db("DELETE FROM Nurses WHERE NurseID=?", (nid,), fetch_all=False)
            self.refresh_nurses()
    
    # ---------- APPOINTMENTS ----------
    def setup_appointments(self):
        tab = self.tabview.tab("Appointments")
        input_frame = ctk.CTkFrame(tab)
        input_frame.pack(fill="x", padx=10, pady=10)
        
        ctk.CTkLabel(input_frame, text="Patient:").grid(row=0, column=0, padx=5, pady=5)
        self.app_patient = ttk.Combobox(input_frame, width=30, state="readonly")
        self.app_patient.grid(row=0, column=1, padx=5, pady=5)
        
        ctk.CTkLabel(input_frame, text="Doctor:").grid(row=0, column=2, padx=5, pady=5)
        self.app_doctor = ttk.Combobox(input_frame, width=30, state="readonly")
        self.app_doctor.grid(row=0, column=3, padx=5, pady=5)
        
        ctk.CTkLabel(input_frame, text="Date (YYYY-MM-DD):").grid(row=1, column=0, padx=5, pady=5)
        self.app_date = ctk.CTkEntry(input_frame, width=20)
        self.app_date.grid(row=1, column=1, padx=5, pady=5)
        self.app_date.insert(0, datetime.today().strftime("%Y-%m-%d"))
        
        ctk.CTkLabel(input_frame, text="Symptoms:").grid(row=1, column=2, padx=5, pady=5)
        self.app_symptoms = ctk.CTkEntry(input_frame, width=40)
        self.app_symptoms.grid(row=1, column=3, padx=5, pady=5)
        
        ctk.CTkButton(input_frame, text="🔍 Check Availability", command=self.check_availability).grid(row=2, column=0, columnspan=4, pady=10)
        
        ctk.CTkLabel(tab, text="Available Time Slots:").pack(anchor="w", padx=10)
        self.slots_list = tk.Listbox(tab, height=8, bg="#2b2b2b", fg="white")
        self.slots_list.pack(fill="x", padx=10, pady=5)
        
        ctk.CTkButton(tab, text="📅 Book Appointment", command=self.book_appointment).pack(pady=5)
        
        self.appointments_tree = ttk.Treeview(tab, columns=("ID","Patient","Doctor","Date","Time","Status","Symptoms"), show="headings")
        for c in ("ID","Patient","Doctor","Date","Time","Status","Symptoms"):
            self.appointments_tree.heading(c, text=c)
        self.appointments_tree.pack(fill="both", expand=True, padx=10, pady=10)
        
        btnf = ctk.CTkFrame(tab)
        btnf.pack(fill="x", padx=10, pady=10)
        ctk.CTkButton(btnf, text="🔄 Refresh", command=self.refresh_appointments).pack(side="left", padx=5)
        ctk.CTkButton(btnf, text="✅ Mark Completed", command=self.mark_completed).pack(side="left", padx=5)
        
        self.refresh_patient_doctor_combos()
        self.refresh_appointments()
    
    def refresh_patient_doctor_combos(self):
        pats = query_db("SELECT PatientID, FullName FROM Patients")
        if pats:
            self.app_patient['values'] = [f"{p[0]} - {p[1]}" for p in pats]
            self.app_patient.current(0)
        else:
            self.app_patient['values'] = ["-- No patients --"]
        docs = query_db("SELECT DoctorID, FullName FROM Doctors")
        if docs:
            self.app_doctor['values'] = [f"{d[0]} - {d[1]}" for d in docs]
            self.app_doctor.current(0)
        else:
            self.app_doctor['values'] = ["-- No doctors --"]
    
    def check_availability(self):
        doc_val = self.app_doctor.get()
        if not doc_val or doc_val.startswith("--"):
            messagebox.showwarning("Warning", "Select a valid doctor")
            return
        doc_id = int(doc_val.split(" - ")[0])
        date = self.app_date.get().strip()
        try:
            dt = datetime.strptime(date, "%Y-%m-%d")
            dow = dt.weekday()
        except:
            messagebox.showerror("Error", "Invalid date format")
            return
        schedule = query_db("SELECT StartTime, EndTime, SlotDuration FROM DoctorAvailability WHERE DoctorID=? AND DayOfWeek=?", (doc_id, dow))
        if not schedule:
            self.slots_list.delete(0, tk.END)
            self.slots_list.insert(tk.END, "Doctor not available on this day")
            return
        start, end, dur = schedule[0]
        booked = query_db("SELECT StartTime FROM Appointments WHERE DoctorID=? AND AppointmentDate=?", (doc_id, date))
        booked_times = [str(b[0]) for b in booked]
        slots = []
        cur_t = datetime.strptime(str(start), "%H:%M:%S")
        end_t = datetime.strptime(str(end), "%H:%M:%S")
        while cur_t < end_t:
            slot = cur_t.strftime("%H:%M")
            if slot not in booked_times:
                slots.append(slot)
            cur_t += timedelta(minutes=dur)
        self.slots_list.delete(0, tk.END)
        for s in slots:
            self.slots_list.insert(tk.END, s)
        if not slots:
            self.slots_list.insert(tk.END, "No available slots")
    
    def book_appointment(self):
        pat_val = self.app_patient.get()
        doc_val = self.app_doctor.get()
        if pat_val.startswith("--") or doc_val.startswith("--"):
            messagebox.showwarning("Warning", "Select valid patient and doctor")
            return
        pat_id = int(pat_val.split(" - ")[0])
        doc_id = int(doc_val.split(" - ")[0])
        date = self.app_date.get().strip()
        symptoms = self.app_symptoms.get().strip()
        sel = self.slots_list.curselection()
        if not sel:
            messagebox.showwarning("Warning", "Select a time slot")
            return
        start_time = self.slots_list.get(sel[0])
        end_time = (datetime.strptime(start_time, "%H:%M") + timedelta(minutes=30)).strftime("%H:%M")
        query_db("INSERT INTO Appointments (PatientID,DoctorID,AppointmentDate,StartTime,EndTime,Symptoms,Status) VALUES (?,?,?,?,?,?,'Scheduled')", (pat_id, doc_id, date, start_time, end_time, symptoms), fetch_all=False)
        messagebox.showinfo("Success", "Appointment booked")
        self.refresh_appointments()
        self.check_availability()
        if self.role in ("Admin","Receptionist"):
            self.update_dashboard()
    
    def refresh_appointments(self):
        for row in self.appointments_tree.get_children():
            self.appointments_tree.delete(row)
        rows = query_db("SELECT a.AppointmentID, p.FullName, d.FullName, a.AppointmentDate, a.StartTime, a.Status, a.Symptoms FROM Appointments a JOIN Patients p ON a.PatientID=p.PatientID JOIN Doctors d ON a.DoctorID=d.DoctorID ORDER BY a.AppointmentDate DESC")
        for r in rows:
            self.appointments_tree.insert("", "end", values=sanitize_row(r))
    
    def mark_completed(self):
        sel = self.appointments_tree.selection()
        if not sel:
            messagebox.showwarning("Warning", "Select an appointment")
            return
        app_id = self.appointments_tree.item(sel[0])['values'][0]
        query_db("UPDATE Appointments SET Status='Completed' WHERE AppointmentID=?", (app_id,), fetch_all=False)
        self.refresh_appointments()
        self.refresh_completed_appointments()
        if self.role in ("Admin","Receptionist"):
            self.update_dashboard()
        messagebox.showinfo("Success", "Marked as completed")
    
    def refresh_completed_appointments(self):
        rows = query_db("SELECT a.AppointmentID, p.FullName, d.FullName, a.AppointmentDate FROM Appointments a JOIN Patients p ON a.PatientID=p.PatientID JOIN Doctors d ON a.DoctorID=d.DoctorID WHERE a.Status='Completed'")
        vals = [f"{r[0]} - {r[1]} with Dr. {r[2]} on {r[3]}" for r in rows]
        if hasattr(self, 'pres_app'):
            self.pres_app['values'] = vals
        if hasattr(self, 'bill_app'):
            self.bill_app['values'] = vals
    
    # ---------- PRESCRIPTIONS ----------
    def setup_prescriptions(self):
        tab = self.tabview.tab("Prescriptions")
        ctk.CTkLabel(tab, text="Select Completed Appointment:").pack(anchor="w", padx=10, pady=5)
        self.pres_app = ttk.Combobox(tab, width=80, state="readonly")
        self.pres_app.pack(padx=10, pady=5)
        ctk.CTkButton(tab, text="📋 Load", command=self.load_prescription).pack(pady=5)
        frame = ctk.CTkFrame(tab)
        frame.pack(fill="both", expand=True, padx=10, pady=10)
        ctk.CTkLabel(frame, text="Diagnosis:").pack(anchor="w")
        self.diag_text = tk.Text(frame, height=4, width=80, bg="#2b2b2b", fg="white")
        self.diag_text.pack(fill="x", padx=5, pady=5)
        ctk.CTkLabel(frame, text="Medicines (with dosage):").pack(anchor="w")
        self.med_text = tk.Text(frame, height=6, width=80, bg="#2b2b2b", fg="white")
        self.med_text.pack(fill="x", padx=5, pady=5)
        ctk.CTkLabel(frame, text="Advice:").pack(anchor="w")
        self.adv_text = tk.Text(frame, height=4, width=80, bg="#2b2b2b", fg="white")
        self.adv_text.pack(fill="x", padx=5, pady=5)
        ctk.CTkLabel(frame, text="Follow-up Date (YYYY-MM-DD):").pack(anchor="w")
        self.follow_entry = ctk.CTkEntry(frame, width=150)
        self.follow_entry.pack(anchor="w", padx=5, pady=2)
        ctk.CTkButton(frame, text="💾 Save Prescription", command=self.save_prescription).pack(pady=10)
        self.refresh_completed_appointments()
    
    def load_prescription(self):
        sel = self.pres_app.get()
        if not sel:
            return
        app_id = int(sel.split(" - ")[0])
        data = query_db("SELECT Diagnosis,Medicines,Advice,FollowUpDate FROM Prescriptions WHERE AppointmentID=?", (app_id,), fetch_all=False)
        self.diag_text.delete("1.0", tk.END)
        self.med_text.delete("1.0", tk.END)
        self.adv_text.delete("1.0", tk.END)
        self.follow_entry.delete(0, tk.END)
        if data:
            self.diag_text.insert("1.0", data[0] or "")
            self.med_text.insert("1.0", data[1] or "")
            self.adv_text.insert("1.0", data[2] or "")
            if data[3]:
                self.follow_entry.insert(0, str(data[3]))
    
    def save_prescription(self):
        sel = self.pres_app.get()
        if not sel:
            messagebox.showwarning("Warning", "Select an appointment")
            return
        app_id = int(sel.split(" - ")[0])
        diag = self.diag_text.get("1.0", tk.END).strip()
        med = self.med_text.get("1.0", tk.END).strip()
        adv = self.adv_text.get("1.0", tk.END).strip()
        follow = self.follow_entry.get().strip() or None
        exists = query_db("SELECT PrescriptionID FROM Prescriptions WHERE AppointmentID=?", (app_id,))
        if exists:
            query_db("UPDATE Prescriptions SET Diagnosis=?, Medicines=?, Advice=?, FollowUpDate=? WHERE AppointmentID=?", (diag, med, adv, follow, app_id), fetch_all=False)
        else:
            query_db("INSERT INTO Prescriptions (AppointmentID, Diagnosis, Medicines, Advice, FollowUpDate) VALUES (?,?,?,?,?)", (app_id, diag, med, adv, follow), fetch_all=False)
        messagebox.showinfo("Success", "Prescription saved")
    
    # ---------- BILLING ----------
    def setup_billing(self):
        tab = self.tabview.tab("Billing")
        ctk.CTkLabel(tab, text="Select Completed Appointment:").pack(anchor="w", padx=10, pady=5)
        self.bill_app = ttk.Combobox(tab, width=80, state="readonly")
        self.bill_app.pack(padx=10, pady=5)
        ctk.CTkButton(tab, text="💰 Load Bill", command=self.load_bill).pack(pady=5)
        frame = ctk.CTkFrame(tab)
        frame.pack(fill="both", expand=True, padx=10, pady=10)
        ctk.CTkLabel(frame, text="Consultation Fee ($):").grid(row=0, column=0, padx=10, pady=5, sticky="w")
        self.fee_label = ctk.CTkLabel(frame, text="0")
        self.fee_label.grid(row=0, column=1, padx=10, pady=5, sticky="w")
        ctk.CTkLabel(frame, text="Additional Charges ($):").grid(row=1, column=0, padx=10, pady=5, sticky="w")
        self.extra_entry = ctk.CTkEntry(frame, width=100)
        self.extra_entry.grid(row=1, column=1, padx=10, pady=5)
        self.extra_entry.bind("<KeyRelease>", lambda e: self.update_total())
        ctk.CTkLabel(frame, text="Total Amount ($):").grid(row=2, column=0, padx=10, pady=5, sticky="w")
        self.total_label = ctk.CTkLabel(frame, text="0")
        self.total_label.grid(row=2, column=1, padx=10, pady=5)
        ctk.CTkLabel(frame, text="Amount Paid ($):").grid(row=3, column=0, padx=10, pady=5, sticky="w")
        self.paid_entry = ctk.CTkEntry(frame, width=100)
        self.paid_entry.grid(row=3, column=1, padx=10, pady=5)
        self.paid_entry.bind("<KeyRelease>", lambda e: self.update_status())
        ctk.CTkLabel(frame, text="Payment Status:").grid(row=4, column=0, padx=10, pady=5, sticky="w")
        self.status_combo = ttk.Combobox(frame, values=["Unpaid","Paid","Partial"], width=15, state="readonly")
        self.status_combo.grid(row=4, column=1, padx=10, pady=5)
        self.status_combo.set("Unpaid")
        ctk.CTkButton(frame, text="💾 Save Bill", command=self.save_bill).grid(row=5, column=0, columnspan=2, pady=20)
        self.refresh_completed_appointments()
    
    def load_bill(self):
        sel = self.bill_app.get()
        if not sel:
            return
        app_id = int(sel.split(" - ")[0])
        fee_row = query_db("SELECT ConsultationFee FROM Doctors d JOIN Appointments a ON d.DoctorID=a.DoctorID WHERE a.AppointmentID=?", (app_id,), fetch_all=False)
        fee = fee_row[0] if fee_row else 0
        self.fee_label.configure(text=str(fee))
        bill = query_db("SELECT AdditionalCharges, TotalAmount, PaidAmount, PaymentStatus FROM Bills WHERE AppointmentID=?", (app_id,), fetch_all=False)
        if bill:
            self.extra_entry.delete(0, tk.END)
            self.extra_entry.insert(0, str(bill[0]))
            self.total_label.configure(text=str(bill[1]))
            self.paid_entry.delete(0, tk.END)
            self.paid_entry.insert(0, str(bill[2]))
            self.status_combo.set(bill[3])
        else:
            self.extra_entry.delete(0, tk.END)
            self.extra_entry.insert(0, "0")
            self.paid_entry.delete(0, tk.END)
            self.paid_entry.insert(0, "0")
            self.status_combo.set("Unpaid")
            self.update_total()
    
    def update_total(self):
        try:
            fee = float(self.fee_label.cget("text"))
            extra = float(self.extra_entry.get().strip() or 0)
            self.total_label.configure(text=str(fee+extra))
        except:
            pass
        self.update_status()
    
    def update_status(self):
        try:
            total = float(self.total_label.cget("text"))
            paid = float(self.paid_entry.get().strip() or 0)
            if paid >= total:
                self.status_combo.set("Paid")
            elif paid > 0:
                self.status_combo.set("Partial")
            else:
                self.status_combo.set("Unpaid")
        except:
            pass
    
    def save_bill(self):
        sel = self.bill_app.get()
        if not sel:
            messagebox.showwarning("Warning", "Select an appointment")
            return
        app_id = int(sel.split(" - ")[0])
        fee = float(self.fee_label.cget("text"))
        extra = float(self.extra_entry.get().strip() or 0)
        total = fee + extra
        paid = float(self.paid_entry.get().strip() or 0)
        status = self.status_combo.get()
        exists = query_db("SELECT BillID FROM Bills WHERE AppointmentID=?", (app_id,))
        if exists:
            query_db("UPDATE Bills SET ConsultationFee=?, AdditionalCharges=?, TotalAmount=?, PaidAmount=?, PaymentStatus=? WHERE AppointmentID=?", (fee, extra, total, paid, status, app_id), fetch_all=False)
        else:
            query_db("INSERT INTO Bills (AppointmentID, ConsultationFee, AdditionalCharges, TotalAmount, PaidAmount, PaymentStatus) VALUES (?,?,?,?,?,?)", (app_id, fee, extra, total, paid, status), fetch_all=False)
        messagebox.showinfo("Success", "Bill saved")
    
    # ---------- BED/WARD MANAGEMENT ----------
    def setup_bed_ward(self):
        tab = self.tabview.tab("Bed/Ward")
        self.ward_tree = ttk.Treeview(tab, columns=("WardID","Ward Name","Type","Total Beds","Occupied","Available"), show="headings")
        self.ward_tree.pack(fill="x", padx=10, pady=5)
        for c in ("WardID","Ward Name","Type","Total Beds","Occupied","Available"):
            self.ward_tree.heading(c, text=c)
        self.bed_tree = ttk.Treeview(tab, columns=("BedID","Ward","Bed Number","Status","Patient"), show="headings")
        self.bed_tree.pack(fill="both", expand=True, padx=10, pady=5)
        for c in ("BedID","Ward","Bed Number","Status","Patient"):
            self.bed_tree.heading(c, text=c)
        btnf = ctk.CTkFrame(tab)
        btnf.pack(fill="x", padx=10, pady=5)
        ctk.CTkButton(btnf, text="🛏️ Admit Patient", command=self.admit_patient).pack(side="left", padx=5)
        ctk.CTkButton(btnf, text="🚪 Discharge Patient", command=self.discharge_patient).pack(side="left", padx=5)
        ctk.CTkButton(btnf, text="🔄 Refresh", command=self.refresh_beds).pack(side="left", padx=5)
        self.refresh_beds()
    
    def refresh_beds(self):
        for row in self.ward_tree.get_children():
            self.ward_tree.delete(row)
        wards = query_db("SELECT WardID, WardName, WardType FROM Wards")
        for w in wards:
            total = query_db("SELECT COUNT(*) FROM Beds WHERE WardID=?", (w[0],))
            occ = query_db("SELECT COUNT(*) FROM Beds WHERE WardID=? AND IsOccupied=1", (w[0],))
            avail = total[0][0] - occ[0][0]
            self.ward_tree.insert("", "end", values=sanitize_row((w[0], w[1], w[2], total[0][0], occ[0][0], avail)))
        for row in self.bed_tree.get_children():
            self.bed_tree.delete(row)
        beds = query_db("SELECT b.BedID, w.WardName, b.BedNumber, CASE WHEN b.IsOccupied=1 THEN 'Occupied' ELSE 'Available' END, (SELECT TOP 1 p.FullName FROM Admissions a JOIN Patients p ON a.PatientID=p.PatientID WHERE a.BedID=b.BedID AND a.DischargeDate IS NULL) FROM Beds b JOIN Wards w ON b.WardID=w.WardID")
        for b in beds:
            self.bed_tree.insert("", "end", values=sanitize_row(b))
    
    def admit_patient(self):
        win = ctk.CTkToplevel(self.root)
        win.title("Admit Patient")
        win.geometry("400x300")
        ctk.CTkLabel(win, text="Patient:").pack(pady=5)
        pat_combo = ttk.Combobox(win, values=[f"{p[0]} - {p[1]}" for p in query_db("SELECT PatientID, FullName FROM Patients")], state="readonly")
        pat_combo.pack(pady=5)
        ctk.CTkLabel(win, text="Bed:").pack(pady=5)
        beds = query_db("SELECT BedID, WardName, BedNumber FROM Beds b JOIN Wards w ON b.WardID=w.WardID WHERE IsOccupied=0")
        bed_combo = ttk.Combobox(win, values=[f"{b[0]} - {b[1]} - {b[2]}" for b in beds], state="readonly")
        bed_combo.pack(pady=5)
        def do_admit():
            pat_val = pat_combo.get()
            bed_val = bed_combo.get()
            if not pat_val or not bed_val:
                messagebox.showerror("Error", "Select patient and bed")
                return
            pat_id = int(pat_val.split(" - ")[0])
            bed_id = int(bed_val.split(" - ")[0])
            query_db("INSERT INTO Admissions (PatientID, BedID) VALUES (?,?)", (pat_id, bed_id), fetch_all=False)
            query_db("UPDATE Beds SET IsOccupied=1 WHERE BedID=?", (bed_id,), fetch_all=False)
            messagebox.showinfo("Success", "Patient admitted")
            win.destroy()
            self.refresh_beds()
        ctk.CTkButton(win, text="Admit", command=do_admit).pack(pady=20)
    
    def discharge_patient(self):
        sel = self.bed_tree.selection()
        if not sel:
            messagebox.showwarning("Warning", "Select an occupied bed")
            return
        bed_id = self.bed_tree.item(sel[0])['values'][0]
        if self.bed_tree.item(sel[0])['values'][3] == "Available":
            messagebox.showwarning("Warning", "Bed is not occupied")
            return
        if messagebox.askyesno("Confirm", "Discharge patient?"):
            query_db("UPDATE Admissions SET DischargeDate=GETDATE() WHERE BedID=? AND DischargeDate IS NULL", (bed_id,), fetch_all=False)
            query_db("UPDATE Beds SET IsOccupied=0 WHERE BedID=?", (bed_id,), fetch_all=False)
            messagebox.showinfo("Success", "Patient discharged")
            self.refresh_beds()
    
    # ---------- LABORATORY ----------
    def setup_laboratory(self):
        tab = self.tabview.tab("Laboratory")
        self.lab_tree = ttk.Treeview(tab, columns=("OrderID","Patient","Test","Ordered Date","Status","Result"), show="headings")
        self.lab_tree.pack(fill="both", expand=True, padx=10, pady=5)
        for c in ("OrderID","Patient","Test","Ordered Date","Status","Result"):
            self.lab_tree.heading(c, text=c)
        btnf = ctk.CTkFrame(tab)
        btnf.pack(fill="x", padx=10, pady=5)
        ctk.CTkButton(btnf, text="➕ New Lab Order", command=self.new_lab_order).pack(side="left", padx=5)
        ctk.CTkButton(btnf, text="🔬 Enter Result", command=self.enter_lab_result).pack(side="left", padx=5)
        self.refresh_lab_orders()
    
    def refresh_lab_orders(self):
        for row in self.lab_tree.get_children():
            self.lab_tree.delete(row)
        rows = query_db("SELECT lo.OrderID, p.FullName, lt.TestName, lo.OrderedDate, lo.Status, lo.ResultValue FROM LabOrders lo JOIN Appointments a ON lo.AppointmentID=a.AppointmentID JOIN Patients p ON a.PatientID=p.PatientID JOIN LabTests lt ON lo.TestID=lt.TestID ORDER BY lo.OrderedDate DESC")
        for r in rows:
            self.lab_tree.insert("", "end", values=sanitize_row(r))
    
    def new_lab_order(self):
        win = ctk.CTkToplevel(self.root)
        win.title("New Lab Order")
        win.geometry("400x300")
        ctk.CTkLabel(win, text="Appointment:").pack(pady=5)
        appts = query_db("SELECT a.AppointmentID, p.FullName FROM Appointments a JOIN Patients p ON a.PatientID=p.PatientID WHERE a.Status IN ('Scheduled','Completed')")
        app_combo = ttk.Combobox(win, values=[f"{a[0]} - {a[1]}" for a in appts], state="readonly")
        app_combo.pack(pady=5)
        ctk.CTkLabel(win, text="Lab Test:").pack(pady=5)
        tests = query_db("SELECT TestID, TestName FROM LabTests")
        test_combo = ttk.Combobox(win, values=[f"{t[0]} - {t[1]}" for t in tests], state="readonly")
        test_combo.pack(pady=5)
        def save_order():
            app_val = app_combo.get()
            test_val = test_combo.get()
            if not app_val or not test_val:
                messagebox.showerror("Error", "Select both fields")
                return
            app_id = int(app_val.split(" - ")[0])
            test_id = int(test_val.split(" - ")[0])
            query_db("INSERT INTO LabOrders (AppointmentID, TestID, Status) VALUES (?,?, 'Pending')", (app_id, test_id), fetch_all=False)
            messagebox.showinfo("Success", "Lab order created")
            win.destroy()
            self.refresh_lab_orders()
        ctk.CTkButton(win, text="Create Order", command=save_order).pack(pady=20)
    
    def enter_lab_result(self):
        sel = self.lab_tree.selection()
        if not sel:
            messagebox.showwarning("Warning", "Select an order")
            return
        order_id = self.lab_tree.item(sel[0])['values'][0]
        win = ctk.CTkToplevel(self.root)
        win.title("Enter Lab Result")
        win.geometry("400x300")
        ctk.CTkLabel(win, text="Result Value:").pack(pady=5)
        result_entry = ctk.CTkEntry(win, width=300)
        result_entry.pack(pady=5)
        ctk.CTkLabel(win, text="Is Normal?").pack(pady=5)
        normal_var = tk.BooleanVar(value=True)
        ctk.CTkCheckBox(win, text="Normal", variable=normal_var).pack(pady=5)
        def save_result():
            result = result_entry.get().strip()
            if not result:
                messagebox.showerror("Error", "Enter result value")
                return
            is_normal = 1 if normal_var.get() else 0
            query_db("UPDATE LabOrders SET ResultValue=?, IsNormal=?, Status='Completed', ResultDate=GETDATE() WHERE OrderID=?", (result, is_normal, order_id), fetch_all=False)
            messagebox.showinfo("Success", "Result saved")
            win.destroy()
            self.refresh_lab_orders()
        ctk.CTkButton(win, text="Save Result", command=save_result).pack(pady=20)
    
    # ---------- PHARMACY ----------
    def setup_pharmacy(self):
        tab = self.tabview.tab("Pharmacy")
        self.medicine_tree = ttk.Treeview(tab, columns=("ID","Medicine","Stock","Price","Reorder Level"), show="headings")
        self.medicine_tree.pack(fill="both", expand=True, padx=10, pady=5)
        for c in ("ID","Medicine","Stock","Price","Reorder Level"):
            self.medicine_tree.heading(c, text=c)
        btnf = ctk.CTkFrame(tab)
        btnf.pack(fill="x", padx=10, pady=5)
        ctk.CTkButton(btnf, text="➕ Add Medicine", command=self.add_medicine).pack(side="left", padx=5)
        ctk.CTkButton(btnf, text="📦 Restock", command=self.restock_medicine).pack(side="left", padx=5)
        ctk.CTkButton(btnf, text="⚠️ Low Stock Alert", command=self.low_stock_alert).pack(side="left", padx=5)
        self.refresh_medicines()
    
    def refresh_medicines(self):
        for row in self.medicine_tree.get_children():
            self.medicine_tree.delete(row)
        rows = query_db("SELECT MedicineID, MedicineName, StockQuantity, Price, ReorderLevel FROM Medicines")
        for r in rows:
            self.medicine_tree.insert("", "end", values=sanitize_row(r))
    
    def add_medicine(self):
        win = ctk.CTkToplevel(self.root)
        win.title("Add Medicine")
        win.geometry("400x400")
        fields = {}
        labels = ["Medicine Name","Stock Quantity","Price","Reorder Level"]
        for i, lbl in enumerate(labels):
            ctk.CTkLabel(win, text=lbl).pack(pady=5)
            e = ctk.CTkEntry(win, width=250)
            e.pack()
            fields[lbl] = e
        def save():
            vals = [fields[l].get().strip() for l in labels]
            if not vals[0]:
                messagebox.showerror("Error", "Name required")
                return
            query_db("INSERT INTO Medicines (MedicineName, StockQuantity, Price, ReorderLevel) VALUES (?,?,?,?)",
                     (vals[0], int(vals[1]) if vals[1] else 0, float(vals[2]) if vals[2] else 0, int(vals[3]) if vals[3] else 10), fetch_all=False)
            messagebox.showinfo("Success", "Medicine added")
            win.destroy()
            self.refresh_medicines()
        ctk.CTkButton(win, text="Save", command=save).pack(pady=20)
    
    def restock_medicine(self):
        sel = self.medicine_tree.selection()
        if not sel:
            messagebox.showwarning("Warning", "Select a medicine")
            return
        med_id = self.medicine_tree.item(sel[0])['values'][0]
        win = ctk.CTkToplevel(self.root)
        win.title("Restock Medicine")
        win.geometry("300x200")
        ctk.CTkLabel(win, text="Quantity to add:").pack(pady=5)
        qty_entry = ctk.CTkEntry(win, width=150)
        qty_entry.pack(pady=5)
        def do_restock():
            try:
                qty = int(qty_entry.get().strip())
                query_db("UPDATE Medicines SET StockQuantity = StockQuantity + ? WHERE MedicineID=?", (qty, med_id), fetch_all=False)
                messagebox.showinfo("Success", f"Added {qty} to stock")
                win.destroy()
                self.refresh_medicines()
            except:
                messagebox.showerror("Error", "Enter a valid number")
        ctk.CTkButton(win, text="Restock", command=do_restock).pack(pady=20)
    
    def low_stock_alert(self):
        low_stock = query_db("SELECT MedicineName, StockQuantity, ReorderLevel FROM Medicines WHERE StockQuantity <= ReorderLevel")
        if low_stock:
            msg = "\n".join([f"{m[0]}: stock {m[1]} (reorder level {m[2]})" for m in low_stock])
            messagebox.showwarning("Low Stock Alert", msg)
        else:
            messagebox.showinfo("Info", "All medicines are sufficiently stocked.")
    
    # ---------- AUDIT LOGS ----------
    def setup_audit_logs(self):
        tab = self.tabview.tab("Audit Logs")
        self.audit_tree = ttk.Treeview(tab, columns=("Timestamp","User","Action","Table","RecordID"), show="headings")
        for c in ("Timestamp","User","Action","Table","RecordID"):
            self.audit_tree.heading(c, text=c)
        self.audit_tree.pack(fill="both", expand=True, padx=10, pady=5)
        btnf = ctk.CTkFrame(tab)
        btnf.pack(fill="x", padx=10, pady=5)
        ctk.CTkButton(btnf, text="🔄 Refresh", command=self.refresh_audit).pack(side="left", padx=5)
        self.refresh_audit()
    
    def refresh_audit(self):
        for row in self.audit_tree.get_children():
            self.audit_tree.delete(row)
        rows = query_db("SELECT TOP 100 Timestamp, Username, Action, TableName, RecordID FROM AuditLog ORDER BY Timestamp DESC")
        for r in rows:
            self.audit_tree.insert("", "end", values=sanitize_row(r))
    
    # ---------- USERS MANAGEMENT ----------
    def setup_users(self):
        tab = self.tabview.tab("Users")
        self.user_tree = ttk.Treeview(tab, columns=("UserID","Username","Role","RelatedID"), show="headings")
        for c in ("UserID","Username","Role","RelatedID"):
            self.user_tree.heading(c, text=c)
        self.user_tree.pack(fill="both", expand=True, padx=10, pady=5)
        btnf = ctk.CTkFrame(tab)
        btnf.pack(fill="x", padx=10, pady=5)
        ctk.CTkButton(btnf, text="➕ Add User", command=self.add_user).pack(side="left", padx=5)
        ctk.CTkButton(btnf, text="🗑️ Delete User", command=self.delete_user).pack(side="left", padx=5)
        self.refresh_users()
    
    def refresh_users(self):
        for row in self.user_tree.get_children():
            self.user_tree.delete(row)
        rows = query_db("SELECT u.UserID, u.Username, r.RoleName, u.RelatedID FROM Users u JOIN Roles r ON u.RoleID=r.RoleID")
        for r in rows:
            self.user_tree.insert("", "end", values=sanitize_row(r))
    
    def add_user(self):
        win = ctk.CTkToplevel(self.root)
        win.title("Add User")
        win.geometry("400x400")
        ctk.CTkLabel(win, text="Username:").pack(pady=5)
        uname = ctk.CTkEntry(win, width=250)
        uname.pack()
        ctk.CTkLabel(win, text="Password:").pack(pady=5)
        pwd = ctk.CTkEntry(win, width=250, show="*")
        pwd.pack()
        ctk.CTkLabel(win, text="Role:").pack(pady=5)
        roles = query_db("SELECT RoleID, RoleName FROM Roles")
        role_combo = ttk.Combobox(win, values=[f"{r[0]} - {r[1]}" for r in roles], state="readonly")
        role_combo.pack()
        ctk.CTkLabel(win, text="Related ID (DoctorID or PatientID, optional):").pack(pady=5)
        rel_id = ctk.CTkEntry(win, width=250)
        rel_id.pack()
        def save():
            u = uname.get().strip()
            p = pwd.get().strip()
            rv = role_combo.get()
            if not u or not p or not rv:
                messagebox.showerror("Error", "All fields required")
                return
            role_id = int(rv.split(" - ")[0])
            rid = rel_id.get().strip()
            rid = int(rid) if rid else None
            query_db("INSERT INTO Users (Username, PasswordHash, RoleID, RelatedID) VALUES (?, ?, ?, ?)", (u, p, role_id, rid), fetch_all=False)
            messagebox.showinfo("Success", "User added")
            win.destroy()
            self.refresh_users()
        ctk.CTkButton(win, text="Save", command=save).pack(pady=20)
    
    def delete_user(self):
        sel = self.user_tree.selection()
        if not sel:
            messagebox.showwarning("Warning", "Select a user")
            return
        user_id = self.user_tree.item(sel[0])['values'][0]
        if messagebox.askyesno("Confirm", "Delete user?"):
            query_db("DELETE FROM Users WHERE UserID=?", (user_id,), fetch_all=False)
            self.refresh_users()
    
    # ---------- DOCTOR SPECIFIC ----------
    def setup_doctor_schedule(self):
        tab = self.tabview.tab("My Schedule")
        self.doc_schedule_tree = ttk.Treeview(tab, columns=("Date","Time","Patient","Symptoms","Status"), show="headings")
        for c in ("Date","Time","Patient","Symptoms","Status"):
            self.doc_schedule_tree.heading(c, text=c)
        self.doc_schedule_tree.pack(fill="both", expand=True, padx=10, pady=5)
        self.refresh_doctor_schedule()
    
    def refresh_doctor_schedule(self):
        for row in self.doc_schedule_tree.get_children():
            self.doc_schedule_tree.delete(row)
        rows = query_db("SELECT AppointmentDate, StartTime, p.FullName, Symptoms, Status FROM Appointments a JOIN Patients p ON a.PatientID=p.PatientID WHERE DoctorID=?", (self.related_id,))
        for r in rows:
            self.doc_schedule_tree.insert("", "end", values=sanitize_row(r))
    
    def setup_patient_history(self):
        tab = self.tabview.tab("Patient History")
        self.history_patient_combo = ttk.Combobox(tab, values=[f"{p[0]} - {p[1]}" for p in query_db("SELECT PatientID, FullName FROM Patients")], state="readonly")
        self.history_patient_combo.pack(pady=5)
        ctk.CTkButton(tab, text="Load History", command=self.load_patient_history).pack(pady=5)
        self.history_tree = ttk.Treeview(tab, columns=("Date","Doctor","Diagnosis","Medicines","Advice"), show="headings")
        for c in ("Date","Doctor","Diagnosis","Medicines","Advice"):
            self.history_tree.heading(c, text=c)
        self.history_tree.pack(fill="both", expand=True, padx=10, pady=5)
    
    def load_patient_history(self):
        sel = self.history_patient_combo.get()
        if not sel:
            return
        pat_id = int(sel.split(" - ")[0])
        for row in self.history_tree.get_children():
            self.history_tree.delete(row)
        rows = query_db("SELECT a.AppointmentDate, d.FullName, p.Diagnosis, p.Medicines, p.Advice FROM Prescriptions p JOIN Appointments a ON p.AppointmentID=a.AppointmentID JOIN Doctors d ON a.DoctorID=d.DoctorID WHERE a.PatientID=?", (pat_id,))
        for r in rows:
            self.history_tree.insert("", "end", values=sanitize_row(r))
    
    # ---------- RECEPTIONIST SPECIFIC ----------
    def setup_admissions(self):
        tab = self.tabview.tab("Admissions")
        self.admissions_tree = ttk.Treeview(tab, columns=("AdmissionID","Patient","Bed","Admit Date","Discharge Date","Diagnosis"), show="headings")
        for c in ("AdmissionID","Patient","Bed","Admit Date","Discharge Date","Diagnosis"):
            self.admissions_tree.heading(c, text=c)
        self.admissions_tree.pack(fill="both", expand=True, padx=10, pady=5)
        self.refresh_admissions()
    
    def refresh_admissions(self):
        for row in self.admissions_tree.get_children():
            self.admissions_tree.delete(row)
        rows = query_db("SELECT a.AdmissionID, p.FullName, b.BedNumber, a.AdmitDate, a.DischargeDate, a.Diagnosis FROM Admissions a JOIN Patients p ON a.PatientID=p.PatientID JOIN Beds b ON a.BedID=b.BedID")
        for r in rows:
            self.admissions_tree.insert("", "end", values=sanitize_row(r))
    
    # ---------- PATIENT SPECIFIC ----------
    def setup_patient_appointments(self):
        tab = self.tabview.tab("My Appointments")
        self.patient_appt_tree = ttk.Treeview(tab, columns=("Date","Doctor","Time","Status","Symptoms"), show="headings")
        for c in ("Date","Doctor","Time","Status","Symptoms"):
            self.patient_appt_tree.heading(c, text=c)
        self.patient_appt_tree.pack(fill="both", expand=True, padx=10, pady=5)
        self.refresh_patient_appointments()
    
    def refresh_patient_appointments(self):
        for row in self.patient_appt_tree.get_children():
            self.patient_appt_tree.delete(row)
        rows = query_db("SELECT AppointmentDate, d.FullName, StartTime, Status, Symptoms FROM Appointments a JOIN Doctors d ON a.DoctorID=d.DoctorID WHERE PatientID=?", (self.related_id,))
        for r in rows:
            self.patient_appt_tree.insert("", "end", values=sanitize_row(r))
    
    def setup_patient_prescriptions(self):
        tab = self.tabview.tab("My Prescriptions")
        self.patient_pres_tree = ttk.Treeview(tab, columns=("Date","Doctor","Diagnosis","Medicines","Advice"), show="headings")
        for c in ("Date","Doctor","Diagnosis","Medicines","Advice"):
            self.patient_pres_tree.heading(c, text=c)
        self.patient_pres_tree.pack(fill="both", expand=True, padx=10, pady=5)
        self.refresh_patient_prescriptions()
    
    def refresh_patient_prescriptions(self):
        for row in self.patient_pres_tree.get_children():
            self.patient_pres_tree.delete(row)
        rows = query_db("SELECT a.AppointmentDate, d.FullName, p.Diagnosis, p.Medicines, p.Advice FROM Prescriptions p JOIN Appointments a ON p.AppointmentID=a.AppointmentID JOIN Doctors d ON a.DoctorID=d.DoctorID WHERE a.PatientID=?", (self.related_id,))
        for r in rows:
            self.patient_pres_tree.insert("", "end", values=sanitize_row(r))
    
    def setup_patient_bills(self):
        tab = self.tabview.tab("My Bills")
        self.patient_bill_tree = ttk.Treeview(tab, columns=("Date","Doctor","Total Amount","Paid","Status"), show="headings")
        for c in ("Date","Doctor","Total Amount","Paid","Status"):
            self.patient_bill_tree.heading(c, text=c)
        self.patient_bill_tree.pack(fill="both", expand=True, padx=10, pady=5)
        self.refresh_patient_bills()
    
    def refresh_patient_bills(self):
        for row in self.patient_bill_tree.get_children():
            self.patient_bill_tree.delete(row)
        rows = query_db("SELECT a.AppointmentDate, d.FullName, b.TotalAmount, b.PaidAmount, b.PaymentStatus FROM Bills b JOIN Appointments a ON b.AppointmentID=a.AppointmentID JOIN Doctors d ON a.DoctorID=d.DoctorID WHERE a.PatientID=?", (self.related_id,))
        for r in rows:
            self.patient_bill_tree.insert("", "end", values=sanitize_row(r))

# ========== RUN ==========
if __name__ == "__main__":
    try:
        test = pyodbc.connect(CONN_STR)
        test.close()
        print("Database connection successful.")
        login = LoginWindow()
    except Exception as e:
        print("Connection error:", e)
        input("Press Enter to exit...")

Connected to database
Database connection successful.
